# Bitcoin: dataset construction

Section 3.1 and Table 1. Starting from the per-source feature tables we

1. keep wallets with at least two incoming and two outgoing transfers,
2. drop wallets whose time intervals are undefined,
3. remove anomalous wallets from the recent sample with an Isolation Forest (5% contamination),
4. assemble the three scenarios of Section 3.2 with the conservative labelling rule of Section 2.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

from bfx.bitcoin import FEATURES
from bfx.modeling import SEED, harmonise

DATA = ROOT / "data" / "bitcoin"
OUT = DATA / "scenarios"
OUT.mkdir(exist_ok=True)

tables = {
    "chainabuse": "chainabuse_metrics.csv",
    "elliptic_fraud": "elliptic_fraud_metrics.csv",
    "elliptic_licit": "elliptic_licit_metrics.csv",
    "recent": "recent_sample_metrics.csv",
}
wallets = {name: pd.read_csv(DATA / "features" / file) for name, file in tables.items()}
counts = pd.DataFrame({"with features": {k: len(v) for k, v in wallets.items()}})

## Filters

The activity filter uses the two degree features. Wallets with a single transfer per counterparty have no inter-arrival time; they are dropped rather than imputed.

In [ ]:
for name, df in wallets.items():
    df = df[(df["total_in_degree"] >= 2) & (df["total_out_degree"] >= 2)]
    counts.loc[name, "activity filter"] = len(df)
    df = df.dropna(subset=FEATURES)
    counts.loc[name, "defined intervals"] = len(df)
    wallets[name] = df

recent = wallets["recent"]
forest = IsolationForest(contamination=0.05, random_state=SEED)
inlier = forest.fit_predict(StandardScaler().fit_transform(recent[FEATURES])) == 1
wallets["recent"] = recent[inlier]

counts["after Isolation Forest"] = counts["defined intervals"]
counts.loc["recent", "after Isolation Forest"] = inlier.sum()
counts.astype(int)

## Scenarios

| Scenario | Fraud | Licit |
|---|---|---|
| 1 | ChainAbuse | recent sample |
| 2 | ChainAbuse | Elliptic++ licit |
| 3 | Elliptic++ illicit | Elliptic++ licit |

Within a scenario a licit wallet that is also reported as fraud is dropped, and a wallet appearing twice is kept once.

In [ ]:
scenarios = {
    1: (["chainabuse"], ["recent"]),
    2: (["chainabuse"], ["elliptic_licit"]),
    3: (["elliptic_fraud"], ["elliptic_licit"]),
}
summary = []
for k, (fraud, licit) in scenarios.items():
    data = harmonise({s: wallets[s] for s in fraud + licit}, fraud_sources=fraud, key="wallet")
    data[["wallet", "source", "y"] + FEATURES].to_csv(OUT / f"scenario_{k}.csv", index=False)
    summary.append({"scenario": k, "fraud": int(data["y"].sum()), "licit": int((data["y"] == 0).sum()),
                    "total": len(data)})
pd.DataFrame(summary).set_index("scenario")

Distinct wallets over all sources after filtering (the working dataset):

In [ ]:
all_wallets = pd.concat(wallets.values())["wallet"]
print(f"{all_wallets.nunique():,} distinct wallets")